# MemoryVerse Empirical Extension Study (v5)
**Manuscript:** *When Do Text Proxies Suffice? A Regime Analysis of Sparse, Dense, and Native-Visual Retrieval Under Proxy Noise*

### Kaggle Execution Instructions:
1. **Accelerator:** Select **GPU T4 x2** or **P100** (Session > Accelerator > GPU).
2. **Internet:** Turn **Internet ON** (Session options > Internet ON) for package installs.
3. **Secrets (Optional):** If running gated models, add `HF_TOKEN` in Add-ons > Secrets.
4. **Execution:** Click **Save Version** -> Choose **Save & Run All (Commit)**.
5. **Output Retrieval:** When the commit completes, navigate to the **Output** tab to download `results_v5_final.zip` and `checkpoint1_native_visual.zip`.

In [ ]:
# 1. Environment Diagnostics & Package Installation
import os
import sys
import subprocess
from pathlib import Path

print("=== Python & Platform Diagnostics ===")
print(f"Python Version: {sys.version}")
print(f"Working Directory: {os.getcwd()}")

# Direct caches to /tmp so model weights never count toward /kaggle/working
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf_cache"
os.environ["TORCH_HOME"] = "/tmp/hf_cache"
Path("/tmp/hf_cache").mkdir(parents=True, exist_ok=True)

# Install requirements
req_candidates = list(Path("/kaggle/input").glob("**/requirements_kaggle.txt")) + [Path("requirements_kaggle.txt")]
if req_candidates and req_candidates[0].exists():
    print(f"Installing dependencies from {req_candidates[0]}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_candidates[0])])
else:
    print("Installing core requirements directly...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "colpali-engine>=0.3.0", "accelerate", "bitsandbytes", "sentence-transformers", "timm", "einops"])

In [ ]:
# 2. GPU Hardware Verification
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    subprocess.run(["nvidia-smi"])
else:
    print("Warning: No GPU detected. Running in CPU fallback mode.")

In [ ]:
# 3. Discover Dataset & Execute Unified Runner (kaggle_run_v5.py)
runner_candidates = list(Path("/kaggle/input").glob("**/kaggle_run_v5.py")) + [Path("kaggle/kaggle_run_v5.py"), Path("kaggle_run_v5.py")]
if not runner_candidates or not runner_candidates[0].exists():
    raise FileNotFoundError("Could not locate kaggle_run_v5.py in /kaggle/input or workspace.")

runner_path = runner_candidates[0]
dataset_root = runner_path.parent.parent
print(f"Executing runner script: {runner_path}")
print(f"Detected dataset root: {dataset_root}")

# Execute full run (--run all executes Tasks 3, 4, 2, 1, 6, 5)
cmd = [sys.executable, str(runner_path), "--run", "all", "--data_dir", str(dataset_root)]
proc = subprocess.run(cmd, text=True)
if proc.returncode != 0:
    print("Runner exited with non-zero code. Inspecting working directory for partial results.")

In [ ]:
# 4. Output Archive Verification
import hashlib
working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./working")
print("=== Output Directory Contents ===")
for f in sorted(working.glob("*.zip")):
    size_mb = f.stat().st_size / (1024 * 1024)
    with open(f, "rb") as fp:
        sha = hashlib.sha256(fp.read()).hexdigest()
    print(f"  {f.name:30} : {size_mb:.2f} MB | SHA-256: {sha}")
print("\nExecution complete. Download the output zip archive from the Kaggle Output tab.")